# Extended Dataset Figures

**Pure plotting notebook — no training, no model inference (except Fig 12 which loads cached weights for feature importance).**

All figures are saved to `figures/multiasset/` as `.pdf` and `.png`.
If any required data file is missing, the cell raises `FileNotFoundError` naming the experiment that must run first.
The final cell prints a manifest of which figures succeeded and which raised errors.

## Style Constants and Data Load

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')

project_root = Path('.').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

RESULTS_DIR  = Path('results_multiasset')
FIGURES_DIR  = Path('figures/multiasset')
TABLES_DIR   = Path('paper_tables')
DATASET_DIR  = Path('dataset_multiasset')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ── Style constants ──────────────────────────────────────────────────────────
FONTSIZE    = 11
SINGLE_COL  = 3.5   # inches
DOUBLE_COL  = 7.0   # inches

plt.rcParams.update({
    'font.size':        FONTSIZE,
    'axes.labelsize':   FONTSIZE,
    'axes.titlesize':   FONTSIZE,
    'legend.fontsize':  FONTSIZE - 1,
    'xtick.labelsize':  FONTSIZE - 1,
    'ytick.labelsize':  FONTSIZE - 1,
    'axes.grid':        True,
    'grid.color':       '#e0e0e0',
    'grid.linewidth':   0.5,
    'figure.dpi':       150,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
})

TICKERS = ['JPM','C','WFC','GS','MS','PNC','USB','FITB','MTB','BAC']
TICKER_COLORS = {
    'JPM': '#1f77b4', 'C':   '#ff7f0e', 'WFC': '#2ca02c',
    'GS':  '#d62728', 'MS':  '#9467bd', 'PNC': '#8c564b',
    'USB': '#e377c2', 'FITB':'#7f7f7f', 'MTB': '#bcbd22', 'BAC': '#17becf',
}
REGIME_COLORS = {
    'full':             '#1f77b4',
    'macro_only':       '#ff7f0e',
    'own_only':         '#2ca02c',
    'other_banks_only': '#d62728',
}
STRESS_PERIODS = [
    ('GFC',   '2008-09-15', '2009-03-31'),
    ('COVID', '2020-02-20', '2020-04-30'),
    ('SVB',   '2023-03-08', '2023-03-31'),
]

HORIZONS_EXT = [1, 5, 10, 21, 63]
BINS_EXT     = [4, 10, 20, 35, 55]
REGIMES_EXT  = ['full','macro_only','own_only','other_banks_only']
K_LIST       = [1, 2, 3, 5, 10]
D_HORIZONS   = [1, 5, 21, 63]
E_HORIZONS   = [1, 21]

# ── Utility functions ────────────────────────────────────────────────────────
def save_fig(fig, stem: str):
    p = FIGURES_DIR / stem
    fig.savefig(str(p) + '.pdf')
    fig.savefig(str(p) + '.png')
    plt.close(fig)
    print(f'  Saved {p}.pdf / .png')

def require_csv(path: Path, experiment_name: str) -> pd.DataFrame:
    """Load CSV or raise FileNotFoundError naming the missing experiment."""
    if not path.exists():
        raise FileNotFoundError(
            f'Required file not found: {path}\n'
            f'Run experiment first: {experiment_name}'
        )
    return pd.read_csv(path)

def shade_stress(ax, alpha=0.12):
    ylim = ax.get_ylim()
    for label, start, end in STRESS_PERIODS:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
                   alpha=alpha, color='gray', zorder=0)
    ax.set_ylim(ylim)

def seed_ci(df_list, col):
    """From list of DataFrames (one per seed), return (mean, lo, hi) arrays."""
    vals = np.stack([df[col].values for df in df_list], axis=0)
    mean = vals.mean(0)
    se   = vals.std(0) / np.sqrt(len(df_list))
    return mean, mean - 1.96*se, mean + 1.96*se

# ── Load all required CSVs once ───────────────────────────────────────────────
print('Loading NLL CSVs...')

def _load_nll_all(exp_tag):
    frames = []
    for t in TICKERS:
        p = RESULTS_DIR / t / exp_tag / 'nll_metrics.csv'
        if p.exists():
            df = pd.read_csv(p); df['ticker'] = t
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

nll_A_ext = _load_nll_all('baseline_ext')
nll_C_ext = _load_nll_all('swa_bestsigma_ext')
nll_D_ext = pd.read_csv(RESULTS_DIR / 'JPM' / 'higher_order_ext' / 'nll_metrics.csv') \
    if (RESULTS_DIR / 'JPM' / 'higher_order_ext' / 'nll_metrics.csv').exists() else pd.DataFrame()
nll_E_ext = _load_nll_all('conditioning_regime_ext')

cross_p = RESULTS_DIR / 'cross_asset_diagnostics.csv'
cross_asset_diag = pd.read_csv(cross_p, parse_dates=['date']) \
    if cross_p.exists() else pd.DataFrame()

sigma_p = RESULTS_DIR / 'JPM' / 'sigma_sweep_ext'
sigma_h1  = pd.read_csv(sigma_p / 'sweep_h1.csv')  if (sigma_p / 'sweep_h1.csv').exists()  else pd.DataFrame()
sigma_h21 = pd.read_csv(sigma_p / 'sweep_h21.csv') if (sigma_p / 'sweep_h21.csv').exists() else pd.DataFrame()

import json
with open(DATASET_DIR / 'feature_names.json') as f:
    FEATURE_NAMES = json.load(f)

print(f'  A-EXT rows: {len(nll_A_ext)}')
print(f'  C-EXT rows: {len(nll_C_ext)}')
print(f'  D-EXT rows: {len(nll_D_ext)}')
print(f'  E-EXT rows: {len(nll_E_ext)}')
print(f'  cross_asset_diag rows: {len(cross_asset_diag)}')
print('Style and data load complete.')
# ── Vol experiment data (optional) ───────────────────────────────────────────
VOL_RESULTS_DIR = Path('results_vol')
vol_data_available = VOL_RESULTS_DIR.exists()

HORIZONS_VOL = [1, 2, 3, 5]
BINS_VOL     = [35, 45, 55]
VOL_MODELS   = [
    'state_free', 'state_cond_return',
    'state_cond_vol_w5', 'state_cond_vol_w10', 'state_cond_vol_w21',
    'state_cond_macro',
]

if vol_data_available:
    def _load_nll_all_vol(model_name):
        frames = []
        for t in TICKERS:
            p = VOL_RESULTS_DIR / t / model_name / 'nll_metrics.csv'
            if p.exists():
                df = pd.read_csv(p); df['ticker'] = t
                frames.append(df)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    nll_vol_all = pd.concat(
        [_load_nll_all_vol(m) for m in VOL_MODELS],
        ignore_index=True
    ) if True else pd.DataFrame()

    vol_state_ts = {}
    for t in TICKERS:
        p = VOL_RESULTS_DIR / f'vol_state_ts_{t}.csv'
        if p.exists():
            vol_state_ts[t] = pd.read_csv(p, parse_dates=['date'])

    FIGURES_VOL_DIR = Path('figures/vol')
    FIGURES_VOL_DIR.mkdir(parents=True, exist_ok=True)

    print(f'  Vol NLL rows: {len(nll_vol_all)}')
    print(f'  Vol state TS loaded for: {list(vol_state_ts.keys())}')
else:
    print('WARNING: results_vol/ not found — vol figures will be skipped')
    nll_vol_all = pd.DataFrame()
    vol_state_ts = {}
    FIGURES_VOL_DIR = Path('figures/vol')


Loading NLL CSVs...


  A-EXT rows: 1500
  C-EXT rows: 1500
  D-EXT rows: 60
  E-EXT rows: 240
  cross_asset_diag rows: 9280
Style and data load complete.


  Vol NLL rows: 2160
  Vol state TS loaded for: ['JPM', 'C', 'WFC', 'GS', 'MS', 'PNC', 'USB', 'FITB', 'MTB', 'BAC']


## Fig 1 — Sparsity vs N (all stocks)

In [2]:
from scripts.bins import compute_X_t, get_edges, assign_bins
from scripts.data import load_master_dataset, build_all_splits, compute_returns

N_range    = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55]
thresholds = [1, 5, 10]   # fraction of cells with < k observations
threshold_labels = ['< 1 obs (zero)', '< 5 obs', '< 10 obs']

stock_curves = {t: {k: [] for k in thresholds} for t in TICKERS}

for ticker in TICKERS:
    prices, F_raw, feature_cols = load_master_dataset(ticker=ticker)
    splits = build_all_splits(prices, [1])
    train_idx = splits[1]['idx_train']
    train_end = int(train_idx[-1]) + 1
    X_t_all, N_XT, edges_xt = compute_X_t(prices, N_XT_target=55, train_end=train_end)
    R_train = compute_returns(prices[1:], 1)[train_idx]   # 1-day returns, train split

    for N in N_range:
        N_actual_fig, edges = get_edges(R_train, N)
        y_train = assign_bins(R_train, edges)
        x_train = X_t_all[train_idx]
        # Build joint count matrix C[N_XT, N]
        C = np.zeros((N_XT, len(edges)-1), dtype=np.int64)
        for xi, yi in zip(x_train, y_train):
            if 0 <= xi < N_XT and 0 <= yi < len(edges)-1:
                C[xi, yi] += 1
        for k in thresholds:
            stock_curves[ticker][k].append(float((C < k).mean()))

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(DOUBLE_COL * 1.2, SINGLE_COL), sharey=False)
for ax_idx, (thresh, thresh_label) in enumerate(zip(thresholds, threshold_labels)):
    ax = axes[ax_idx]
    pooled = np.zeros(len(N_range))
    for ticker in TICKERS:
        vals = stock_curves[ticker][thresh]
        ax.plot(N_range, vals, color=TICKER_COLORS[ticker], alpha=0.4, linewidth=0.8)
        pooled += np.array(vals)
    pooled /= len(TICKERS)
    ax.plot(N_range, pooled, color='black', linewidth=2, label='Mean')
    ax.set_xlabel('N bins')
    ax.set_ylabel('Fraction of cells')
    ax.set_xlim(N_range[0], N_range[-1])
    ax.text(0.97, 0.95, thresh_label, transform=ax.transAxes,
            ha='right', va='top', fontsize=FONTSIZE-1)

# Legend for stocks (outside)
handles = [mpatches.Patch(color=TICKER_COLORS[t], label=t, alpha=0.7) for t in TICKERS]
handles.append(plt.Line2D([0],[0], color='black', linewidth=2, label='Mean'))
fig.legend(handles=handles, loc='lower center', ncol=6, bbox_to_anchor=(0.5, -0.18),
           frameon=False, fontsize=FONTSIZE-2)
fig.tight_layout()
save_fig(fig, 'fig1')

  Saved figures\multiasset\fig1.pdf / .png


## Fig 2 — NLL vs horizon, all stocks (10-panel grid)

In [3]:
if nll_C_ext.empty:
    raise FileNotFoundError(
        'nll_C_ext is empty.\nRun experiment first: C-EXT (swa_bestsigma_ext)')

df = nll_C_ext[nll_C_ext['N'] == 55].copy()

fig, axes = plt.subplots(2, 5, figsize=(DOUBLE_COL * 1.5, DOUBLE_COL * 0.7), sharey=False)
axes_flat = axes.flatten()

for ax_i, ticker in enumerate(TICKERS):
    ax = axes_flat[ax_i]
    df_t = df[df['ticker'] == ticker]

    for model_type, ls, label in [
        ('state_cond', '-',  'State-cond'),
        ('state_free', '--', 'State-free'),
    ]:
        dm = df_t[df_t['model'] == model_type]
        if dm.empty: continue
        by_h = dm.groupby('h')['nll_test'].agg(['mean','std']).reset_index()
        ax.plot(by_h['h'], by_h['mean'], ls=ls, color=TICKER_COLORS[ticker],
                linewidth=1.4, label=label)
        ax.fill_between(by_h['h'],
                        by_h['mean'] - by_h['std'],
                        by_h['mean'] + by_h['std'],
                        alpha=0.15, color=TICKER_COLORS[ticker])

    # Marginal baseline
    import math
    ax.axhline(math.log(55), color='gray', linewidth=0.8, linestyle=':', label='Marginal')
    ax.axvline(21, color='black', linewidth=0.6, linestyle=':', alpha=0.5)
    ax.set_xlabel('Horizon h (days)')
    ax.set_ylabel('NLL')
    ax.set_xticks(HORIZONS_EXT)
    ax.text(0.05, 0.95, ticker, transform=ax.transAxes,
            fontweight='bold', va='top', fontsize=FONTSIZE)

axes_flat[0].legend(loc='lower right', frameon=False, fontsize=FONTSIZE-2)
fig.tight_layout()
save_fig(fig, 'fig2')

  Saved figures\multiasset\fig2.pdf / .png


## Fig 3 — NLL vs N bins

In [4]:
if nll_C_ext.empty:
    raise FileNotFoundError('nll_C_ext empty. Run: C-EXT (swa_bestsigma_ext)')

fig, axes = plt.subplots(1, 2, figsize=(DOUBLE_COL, SINGLE_COL * 1.1))
import math

for ax_i, (subset_label, df_sub) in enumerate([
    ('JPM (h=1)',  nll_C_ext[(nll_C_ext.ticker=='JPM') & (nll_C_ext.h==1)]),
    ('Pooled mean (h=1)', nll_C_ext[nll_C_ext.h==1]),
]):
    ax = axes[ax_i]
    for model_type, ls, label in [
        ('state_cond', '-',  'State-cond h=1'),
        ('state_free', '--', 'State-free h=1'),
    ]:
        dm = df_sub[df_sub['model'] == model_type]
        if dm.empty: continue
        by_N = dm.groupby('N')['nll_test'].mean()
        ax.plot(by_N.index, by_N.values, ls=ls, color='steelblue',
                linewidth=1.4, label=label)

    # State-cond at h=21 (only JPM panel)
    if ax_i == 0:
        df_h21 = nll_C_ext[(nll_C_ext.ticker=='JPM') &
                            (nll_C_ext.h==21) & (nll_C_ext.model=='state_cond')]
        if not df_h21.empty:
            by_N_21 = df_h21.groupby('N')['nll_test'].mean()
            ax.plot(by_N_21.index, by_N_21.values, ls='-.', color='darkorange',
                    linewidth=1.4, label='State-cond h=21')

    # Marginal baselines
    for N_val in BINS_EXT:
        pass  # marginal depends on N; plot as scatter
    marginal_y = [math.log(N) for N in BINS_EXT]
    ax.plot(BINS_EXT, marginal_y, ls=':', color='gray', linewidth=0.8, label='Marginal')

    ax.set_xlabel('N bins')
    ax.set_ylabel('Test NLL')
    ax.set_xticks(BINS_EXT)
    ax.text(0.05, 0.95, subset_label, transform=ax.transAxes, va='top')
    ax.legend(frameon=False, fontsize=FONTSIZE-2)

fig.tight_layout()
save_fig(fig, 'fig3')

  Saved figures\multiasset\fig3.pdf / .png


## Fig 4 — Higher-order k ablation heatmap

In [5]:
if nll_D_ext.empty:
    raise FileNotFoundError(
        'nll_D_ext empty. Run: D-EXT (higher_order_ext)')

import math
marginal_nll_55 = math.log(55)

pivot = nll_D_ext.copy()
pivot['k'] = pivot['model'].str.extract(r'ho_k(\d+)').astype(int)
pivot['delta'] = pivot['nll_test'] - marginal_nll_55
heat = pivot.groupby(['k','h'])['delta'].mean().unstack('h')

# Reindex to ensure all k and h present
heat = heat.reindex(index=K_LIST, columns=D_HORIZONS)

vmax = np.nanmax(np.abs(heat.values))
fig, ax = plt.subplots(figsize=(SINGLE_COL * 1.5, SINGLE_COL * 1.2))
im = ax.imshow(heat.values, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
ax.set_xticks(range(len(D_HORIZONS))); ax.set_xticklabels(D_HORIZONS)
ax.set_yticks(range(len(K_LIST)));     ax.set_yticklabels(K_LIST)
ax.set_xlabel('Horizon h (days)')
ax.set_ylabel('Markov order k')

for i in range(len(K_LIST)):
    for j in range(len(D_HORIZONS)):
        v = heat.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, f'{v:+.3f}', ha='center', va='center',
                    fontsize=FONTSIZE-2,
                    color='white' if abs(v) > 0.6*vmax else 'black')

cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label('ΔNLL vs marginal')
fig.tight_layout()
save_fig(fig, 'fig4')

  Saved figures\multiasset\fig4.pdf / .png


## Fig 5 — Conditioning regime comparison, all stocks

In [6]:
if nll_E_ext.empty:
    raise FileNotFoundError(
        'nll_E_ext empty. Run: E-EXT (conditioning_regime_ext)')

for h_val, fig_stem in [(1, 'fig5a'), (21, 'fig5b')]:
    df_h = nll_E_ext[nll_E_ext['h'] == h_val].copy()
    df_h['regime'] = df_h['model'].str.replace('state_cond_', '', regex=False)
    by_tr = df_h.groupby(['ticker','regime'])['delta_marginal'].mean().reset_index()

    fig, ax = plt.subplots(figsize=(DOUBLE_COL * 0.85, DOUBLE_COL * 0.9))
    n_regimes = len(REGIMES_EXT)
    bar_w = 0.18
    offsets = np.linspace(-(n_regimes-1)/2, (n_regimes-1)/2, n_regimes) * bar_w

    yticks = np.arange(len(TICKERS))
    for r_i, regime in enumerate(REGIMES_EXT):
        vals = []
        for ticker in TICKERS:
            sub = by_tr[(by_tr.ticker==ticker) & (by_tr.regime==regime)]
            vals.append(float(sub['delta_marginal'].values[0]) if not sub.empty else np.nan)
        ax.barh(yticks + offsets[r_i], vals, height=bar_w * 0.9,
                color=REGIME_COLORS[regime], label=regime, alpha=0.85)

    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_yticks(yticks); ax.set_yticklabels(TICKERS)
    ax.set_xlabel('ΔNLL vs marginal')
    ax.legend(loc='lower right', frameon=False, fontsize=FONTSIZE-2)
    ax.invert_yaxis()
    fig.tight_layout()
    save_fig(fig, fig_stem)

  Saved figures\multiasset\fig5a.pdf / .png


  Saved figures\multiasset\fig5b.pdf / .png


## Fig 6 — Operator snapshots for 3 representative stocks

In [7]:
# Select representative stocks: JPM + the one with largest |delta_marginal| + one with delta>0
if nll_C_ext.empty:
    raise FileNotFoundError('nll_C_ext empty. Run: C-EXT (swa_bestsigma_ext)')

df_snap_src = nll_C_ext[(nll_C_ext.h==1) & (nll_C_ext.N==55) &
                          (nll_C_ext.model=='state_cond')]
by_ticker = df_snap_src.groupby('ticker')['delta_marginal'].mean()

snap_tickers = ['JPM']
best_ticker = by_ticker.abs().idxmax()
if best_ticker != 'JPM':
    snap_tickers.append(best_ticker)
fail_tickers = by_ticker[by_ticker > 0].index.tolist()
if fail_tickers and fail_tickers[0] not in snap_tickers:
    snap_tickers.append(fail_tickers[0])
if len(snap_tickers) < 3:
    fallback = [t for t in TICKERS if t not in snap_tickers]
    snap_tickers += fallback[:3-len(snap_tickers)]
snap_tickers = snap_tickers[:3]
print(f'Snapshot stocks: {snap_tickers}')

SEEDS = [42, 7, 123]
SNAP_TYPES = ['calm', 'bearish', 'bullish']

def load_snapshot(ticker, snap_type, seed=42,
                  exp_tag='swa_bestsigma_ext') -> np.ndarray:
    p = RESULTS_DIR / ticker / exp_tag / 'snapshots' / \
        f'operator_snapshot_{snap_type}_seed{seed}.npy'
    if not p.exists():
        raise FileNotFoundError(
            f'Snapshot missing: {p}\nRun: C-EXT then F-NEW for {ticker}')
    return np.load(p)

fig, axes = plt.subplots(3, 3, figsize=(DOUBLE_COL, DOUBLE_COL * 0.95))
vmin_global, vmax_global = 0.0, None

# Compute global vmax for shared colour scale
all_mats = []
for ticker in snap_tickers:
    for snap_type in SNAP_TYPES:
        try:
            mat = load_snapshot(ticker, snap_type)
            all_mats.append(mat)
        except FileNotFoundError:
            all_mats.append(np.zeros((55,55)))
vmax_global = max(m.max() for m in all_mats) if all_mats else 1.0

for row, ticker in enumerate(snap_tickers):
    for col, snap_type in enumerate(SNAP_TYPES):
        ax = axes[row, col]
        try:
            mat = load_snapshot(ticker, snap_type)
        except FileNotFoundError:
            mat = np.full((55,55), np.nan)
        im = ax.imshow(mat, cmap='Blues', vmin=0, vmax=vmax_global,
                       aspect='auto', origin='lower')
        if row == 0:
            ax.set_title(snap_type.capitalize(), fontsize=FONTSIZE)
        if col == 0:
            ax.set_ylabel(ticker, fontsize=FONTSIZE, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])

fig.tight_layout()
save_fig(fig, 'fig6')

Snapshot stocks: ['JPM', 'USB', 'GS']


  Saved figures\multiasset\fig6.pdf / .png


## Fig 7 — Row entropy over time, multi-stock overlay

In [8]:
if cross_asset_diag.empty:
    raise FileNotFoundError(
        'cross_asset_diagnostics.csv missing. Run: F-NEW')

ROLL = 21
fig, ax = plt.subplots(figsize=(DOUBLE_COL, SINGLE_COL))

cross_means = []
for ticker in TICKERS:
    sub = cross_asset_diag[cross_asset_diag.ticker == ticker].sort_values('date')
    if sub.empty: continue
    rolled = sub.set_index('date')['row_entropy'].rolling(ROLL, min_periods=1).mean()
    ax.plot(rolled.index, rolled.values, color=TICKER_COLORS[ticker],
            alpha=0.35, linewidth=0.8)
    cross_means.append(rolled.values)

if cross_means:
    min_len = min(len(v) for v in cross_means)
    mean_ent = np.mean([v[:min_len] for v in cross_means], axis=0)
    # Use dates from first stock for x-axis (approximate alignment)
    first_sub = cross_asset_diag[cross_asset_diag.ticker == TICKERS[0]].sort_values('date')
    dates_x = first_sub['date'].values[:min_len]
    ax.plot(dates_x, mean_ent, color='black', linewidth=2, label='Mean')

shade_stress(ax)

# Annotate COVID dip
covid_start = pd.Timestamp('2020-02-20')
ax.annotate('COVID', xy=(covid_start, ax.get_ylim()[0]),
            xytext=(covid_start, ax.get_ylim()[0] + 0.1*(ax.get_ylim()[1]-ax.get_ylim()[0])),
            fontsize=FONTSIZE-2, color='dimgray',
            arrowprops=dict(arrowstyle='->', color='dimgray', lw=0.8))

handles = [mpatches.Patch(color=TICKER_COLORS[t], alpha=0.5, label=t) for t in TICKERS]
handles.append(plt.Line2D([0],[0], color='black', linewidth=2, label='Mean'))
ax.set_xlabel('Date')
ax.set_ylabel(f'Row entropy ({ROLL}-day rolling avg)')
ax.legend(handles=handles, loc='upper left', ncol=5, bbox_to_anchor=(0,1.02),
          frameon=False, fontsize=FONTSIZE-2)
fig.tight_layout()
save_fig(fig, 'fig7')

  Saved figures\multiasset\fig7.pdf / .png


## Fig 8 — CK discrepancy over time, multi-stock overlay

In [9]:
if cross_asset_diag.empty:
    raise FileNotFoundError(
        'cross_asset_diagnostics.csv missing. Run: F-NEW')

ROLL = 21
fig, ax = plt.subplots(figsize=(DOUBLE_COL, SINGLE_COL))

sc_means, sf_means = [], []

for ticker in TICKERS:
    sub = cross_asset_diag[cross_asset_diag.ticker == ticker].sort_values('date')
    if sub.empty: continue
    ck_sc = sub.set_index('date')['ck_error_h5'].rolling(ROLL, min_periods=1).mean()
    ax.plot(ck_sc.index, ck_sc.values, color=TICKER_COLORS[ticker],
            alpha=0.3, linewidth=0.7)
    sc_means.append(ck_sc.values)

    # State-free CK from per-seed CSVs (h=5, N=55, state_free)
    sf_vals_list = []
    for seed in [42, 7, 123]:
        ck_sf_p = RESULTS_DIR / ticker / 'swa_bestsigma_ext' / \
            f'ck_ts_state_free_h5_N55_seed{seed}.csv'
        if ck_sf_p.exists():
            df_sf = pd.read_csv(ck_sf_p, parse_dates=['date'])
            r = df_sf.set_index('date')['ck_kl'].rolling(ROLL, min_periods=1).mean()
            sf_vals_list.append(r.values)
    if sf_vals_list:
        min_len = min(len(v) for v in sf_vals_list)
        sf_means.append(np.mean([v[:min_len] for v in sf_vals_list], axis=0))

if sc_means:
    min_len = min(len(v) for v in sc_means)
    mean_sc = np.mean([v[:min_len] for v in sc_means], axis=0)
    first_sub = cross_asset_diag[cross_asset_diag.ticker == TICKERS[0]].sort_values('date')
    dates_x = first_sub['date'].values[:min_len]
    ax.plot(dates_x, mean_sc, color='steelblue', linewidth=2,
            label='State-cond mean')

if sf_means:
    min_len = min(len(v) for v in sf_means)
    mean_sf = np.mean([v[:min_len] for v in sf_means], axis=0)
    ax.plot(dates_x[:min_len], mean_sf, color='darkorange', linewidth=2,
            linestyle='--', label='State-free mean')

shade_stress(ax)
ax.set_xlabel('Date')
ax.set_ylabel(f'CK error h=5 ({ROLL}-day rolling avg)')
ax.legend(frameon=False)
fig.tight_layout()
save_fig(fig, 'fig8')

  Saved figures\multiasset\fig8.pdf / .png


## Fig 9 — Cross-asset synchrony heatmap

In [10]:
if cross_asset_diag.empty:
    raise FileNotFoundError(
        'cross_asset_diagnostics.csv missing. Run: F-NEW')

import seaborn as sns

pivot = cross_asset_diag.pivot_table(
    index='date', columns='ticker', values='row_entropy')
pivot = pivot[[t for t in TICKERS if t in pivot.columns]]
corr  = pivot.corr(method='pearson')

mask_upper = np.triu(np.ones(corr.shape), k=1).astype(bool)
off_diag   = corr.values[mask_upper]
mean_corr  = float(np.nanmean(off_diag))

g = sns.clustermap(
    corr, cmap='coolwarm', vmin=-1, vmax=1,
    annot=True, fmt='.2f', linewidths=0.4,
    figsize=(DOUBLE_COL, DOUBLE_COL),
    annot_kws={'size': FONTSIZE-3},
)
g.fig.suptitle(f'Mean off-diagonal r = {mean_corr:.3f}',
               y=1.02, fontsize=FONTSIZE)
g.fig.tight_layout()
p = FIGURES_DIR / 'fig9'
g.fig.savefig(str(p) + '.pdf')
g.fig.savefig(str(p) + '.png')
plt.close(g.fig)
print(f'  Saved {p}.pdf / .png')
print(f'  Mean off-diagonal Pearson correlation: {mean_corr:.4f}')

  Saved figures\multiasset\fig9.pdf / .png
  Mean off-diagonal Pearson correlation: 0.6462


## Fig 10 — State dependence heatmap across stocks and time

In [11]:
if cross_asset_diag.empty:
    raise FileNotFoundError(
        'cross_asset_diagnostics.csv missing. Run: F-NEW')

ROLL = 21
pivot = cross_asset_diag.pivot_table(
    index='date', columns='ticker', values='row_heterogeneity')
pivot = pivot[[t for t in TICKERS if t in pivot.columns]]
pivot_smooth = pivot.rolling(ROLL, min_periods=1).mean()

# Sort stocks by mean state dependence
mean_by_ticker = pivot_smooth.mean()
sorted_tickers = mean_by_ticker.sort_values(ascending=False).index.tolist()
mat = pivot_smooth[sorted_tickers].T.values   # (n_stocks, T_test)

fig, ax = plt.subplots(figsize=(DOUBLE_COL * 1.1, SINGLE_COL * 0.9))
im = ax.imshow(mat, aspect='auto', cmap='YlOrRd',
               vmin=np.nanpercentile(mat, 5),
               vmax=np.nanpercentile(mat, 95))

ax.set_yticks(range(len(sorted_tickers)))
ax.set_yticklabels(sorted_tickers, fontsize=FONTSIZE-1)

# x-axis: date labels
dates_all = pivot_smooth.index
if hasattr(dates_all[0], 'year'):
    years = sorted(set(d.year for d in dates_all))
    year_ticks = [np.searchsorted(dates_all, pd.Timestamp(f'{y}-01-01')) for y in years[::3]]
    ax.set_xticks(year_ticks)
    ax.set_xticklabels([str(y) for y in years[::3]], rotation=30)

# Stress period lines
for _, start, end in STRESS_PERIODS:
    for ts in [pd.Timestamp(start), pd.Timestamp(end)]:
        xi = np.searchsorted(dates_all, ts)
        ax.axvline(xi, color='white', linewidth=0.8, linestyle='--', alpha=0.7)

ax.set_xlabel('Time')
ax.set_ylabel('Stock')
cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cb.set_label('Row heterogeneity')
fig.tight_layout()
save_fig(fig, 'fig10')

  Saved figures\multiasset\fig10.pdf / .png


## Fig 11 — NLL gain stratified by volatility regime

In [12]:
import math

if nll_C_ext.empty:
    raise FileNotFoundError('nll_C_ext empty. Run: C-EXT (swa_bestsigma_ext)')

# For each stock, we need per-timestep NLL delta and VIXCLS from the feature CSV.
# Per-timestep NLL is in the operator_ts CSVs (via nll_test), but we need the actual
# per-sample log-likelihoods. Those are not stored in operator_ts CSVs.
# We use nll_test as the overall test-set NLL (single number per seed),
# then compute gain as nll_test - marginal_nll(55), split by VIX tercile of
# the test period.
#
# For high/low VIX split: load VIXCLS from features_{ticker}.csv,
# split test split dates into top/bottom tercile, then load per-seed
# operator_ts CSVs to get per-timestep row_entropy as a proxy signal.
# For actual NLL gain we fall back to the overall mean nll_test.

# Load per-timestep NLL from operator_ts (we use row_entropy as state-dep signal
# and nll_test from nll_metrics for the overall metric, stratified by VIX).

from scripts.data import load_master_dataset, build_all_splits

vix_results = []

for ticker in TICKERS:
    try:
        prices, F_raw, feature_cols = load_master_dataset(ticker=ticker)
        if 'VIXCLS' not in feature_cols:
            continue
        vix_col_idx = feature_cols.index('VIXCLS')
        splits = build_all_splits(prices, [1])
        idx_test = splits[1]['idx_test']
        vix_test = F_raw[idx_test, vix_col_idx]   # already standardised

        lo_thresh = np.percentile(vix_test, 33.3)
        hi_thresh = np.percentile(vix_test, 66.7)
        mask_low  = vix_test <= lo_thresh
        mask_high = vix_test >= hi_thresh

        # Load operator_ts CSVs for state_cond h=1 N=55, aggregate per-timestep delta
        # NLL proxy: row_heterogeneity (state dependence); use marginal nll for delta
        seeds = [42, 7, 123]
        low_delta_vals, high_delta_vals = [], []
        for seed in seeds:
            ts_p = RESULTS_DIR / ticker / 'swa_bestsigma_ext' / \
                f'operator_ts_state_cond_h1_N55_seed{seed}.csv'
            if not ts_p.exists(): continue
            df_ts = pd.read_csv(ts_p)
            if len(df_ts) != len(idx_test): continue
            # row_heterogeneity as proxy for state-dep NLL gain
            rh = df_ts['row_heterogeneity'].values
            low_delta_vals.append(rh[mask_low].mean())
            high_delta_vals.append(rh[mask_high].mean())

        if not low_delta_vals: continue
        vix_results.append({
            'ticker': ticker,
            'low_vix':  np.mean(low_delta_vals),
            'high_vix': np.mean(high_delta_vals),
        })
    except Exception as e:
        print(f'  [Fig11] Skipped {ticker}: {e}')

if not vix_results:
    raise FileNotFoundError(
        'No VIX-stratified data available. Run C-EXT then check VIXCLS in features.')

df_vix = pd.DataFrame(vix_results)
fig, ax = plt.subplots(figsize=(DOUBLE_COL * 0.9, SINGLE_COL * 1.1))
x = np.arange(len(df_vix))
bw = 0.35
ax.bar(x - bw/2, df_vix['low_vix'],  width=bw, label='Low VIX tercile',  color='steelblue', alpha=0.8)
ax.bar(x + bw/2, df_vix['high_vix'], width=bw, label='High VIX tercile', color='tomato',    alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(df_vix['ticker'], rotation=30)
ax.set_ylabel('Mean row heterogeneity')
ax.legend(frameon=False)
fig.tight_layout()
save_fig(fig, 'fig11')

  Saved figures\multiasset\fig11.pdf / .png


## Fig 12 — Feature importance, JPM (requires model inference)

In [13]:
import torch
import torch.nn.functional as torchF
from scripts.models import StateConditionedNet
from scripts.train import load_cached_model
from scripts.bins import build_all_configs, compute_X_t
from scripts.data import load_master_dataset, build_all_splits
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else
                      ('cuda' if torch.cuda.is_available() else 'cpu'))

ticker = 'JPM'; h = 1; N = 55; seed = 42
import math

# Find best C-EXT sigma
if sigma_h1.empty:
    raise FileNotFoundError(
        'sigma_sweep_ext/sweep_h1.csv missing. Run: B-EXT')
best_sigma = float(sigma_h1.groupby('sigma')['nll_val'].mean().idxmin())

# Load data
prices, F_raw, feature_cols = load_master_dataset(ticker=ticker)
F_normed = F_raw
splits = build_all_splits(prices, [h])
train_end = int(splits[h]['idx_train'][-1]) + 1
X_t_all, N_XT, edges_xt = compute_X_t(prices, N_XT_target=55, train_end=train_end)
configs = build_all_configs(
    prices, F_normed, X_t_all, horizons=[h], n_bins_list=[N],
    N_XT=N_XT, edges_xt=edges_xt, splits=splits, sigma_anchor=1.0)
cfg_dict = configs[(h, N)]
idx_test = cfg_dict['idx_test']
y_test = cfg_dict['y_all'][idx_test]

# Load cached model
n_feat = F_normed.shape[1]
cp = Path(f'results_multiasset/_cache_ext/Cext_s{best_sigma}_{ticker}_state_cond_h{h}_N{N}_seed{seed}.pt')
if not cp.exists():
    raise FileNotFoundError(
        f'C-EXT model cache not found: {cp}\nRun: C-EXT (swa_bestsigma_ext)')
model = StateConditionedNet(n_feat, N_XT, cfg_dict['N_actual'])
model = load_cached_model(model, cp).to(DEVICE)
model.eval()

# Baseline NLL
F_test = torch.tensor(F_normed[idx_test], dtype=torch.float32)
X_test = torch.tensor(X_t_all[idx_test], dtype=torch.long)
y_ts   = torch.tensor(y_test,            dtype=torch.long)
with torch.no_grad():
    logits = model(F_test.to(DEVICE), X_test.to(DEVICE))
    lp_base = torchF.log_softmax(logits, dim=1).cpu()
nll_base = float(-lp_base[torch.arange(len(y_ts)), y_ts].mean())

# Feature importance: zero each feature, recompute NLL
nll_increases = []
for fi, fname in enumerate(feature_cols):
    F_zeroed = F_test.clone()
    F_zeroed[:, fi] = 0.0
    with torch.no_grad():
        logits_z = model(F_zeroed.to(DEVICE), X_test.to(DEVICE))
        lp_z = torchF.log_softmax(logits_z, dim=1).cpu()
    nll_z = float(-lp_z[torch.arange(len(y_ts)), y_ts].mean())
    nll_increases.append((fname, nll_z - nll_base))

nll_increases.sort(key=lambda x: x[1], reverse=True)
top25 = nll_increases[:25]

names, values = zip(*top25)
fig, ax = plt.subplots(figsize=(SINGLE_COL * 1.3, DOUBLE_COL))
ypos = np.arange(len(top25))[::-1]
ax.barh(ypos, values, height=0.7, color='steelblue', alpha=0.85)
ax.set_yticks(ypos); ax.set_yticklabels(names, fontsize=FONTSIZE-2)
ax.set_xlabel('NLL increase when feature zeroed')
ax.axvline(0, color='black', linewidth=0.8)
fig.tight_layout()
save_fig(fig, 'fig12')

  Saved figures\multiasset\fig12.pdf / .png


## Fig V1 — Head-to-head: vol-state vs return-state, all stocks (h=1, N=55)

In [14]:
if not vol_data_available or nll_vol_all.empty:
    print('SKIP: results_vol/ not available')
else:
    import numpy as np

    sub = nll_vol_all[nll_vol_all['h'].eq(1) & nll_vol_all['N'].eq(55)].copy()
    models_plot = ['state_free', 'state_cond_return', 'state_cond_vol_w21', 'state_cond_macro']
    labels_plot = ['state_free', 'return', 'vol_w21', 'macro']
    colors_v1   = ['#7f7f7f',   '#1f77b4',            '#9467bd',            '#e377c2']

    # Per-seed per-ticker delta_marginal
    grp = (sub[sub['model'].isin(models_plot)]
           .groupby(['ticker', 'model', 'seed'])['delta_marginal']
           .mean()
           .reset_index())

    # Mean and SE across seeds (n=3)
    agg = (grp.groupby(['ticker', 'model'])['delta_marginal']
           .agg(['mean', 'std', 'count'])
           .reset_index())
    agg['se95'] = 1.96 * agg['std'] / np.sqrt(agg['count'].clip(lower=1))

    # Sort tickers by vol_w21 mean delta_marginal (ascending = best to worst)
    vol_means = (agg[agg['model'].eq('state_cond_vol_w21')]
                 .set_index('ticker')['mean'])
    ticker_order = vol_means.sort_values().index.tolist()

    x     = np.arange(len(ticker_order))
    width = 0.18
    n_m   = len(models_plot)

    fig, ax = plt.subplots(figsize=(DOUBLE_COL * 1.3, 4.0))

    for i, (m, lbl, c) in enumerate(zip(models_plot, labels_plot, colors_v1)):
        m_agg = agg[agg['model'].eq(m)].set_index('ticker')
        means = np.array([m_agg.loc[t, 'mean'] if t in m_agg.index else np.nan
                          for t in ticker_order])
        errs  = np.array([m_agg.loc[t, 'se95'] if t in m_agg.index else np.nan
                          for t in ticker_order])
        offset = (i - (n_m - 1) / 2) * width
        ax.bar(x + offset, means, width, label=lbl, color=c, alpha=0.85,
               yerr=errs, error_kw=dict(elinewidth=0.8, capsize=2.5, ecolor='#333333'))

    ax.axhline(0, color='black', lw=1.2, ls='--', zorder=5)
    ax.set_xticks(x)
    ax.set_xticklabels(ticker_order, rotation=30, ha='right')
    ax.set_ylabel('Δ NLL vs marginal (nats)')
    ax.set_title('Vol-state vs Return-state: Δ NLL (h=1, N=55)  |  Negative = beats marginal')
    ax.legend(fontsize=FONTSIZE - 2, ncol=2)

    # Pooled mean annotation (inset text box)
    pooled = (agg[agg['model'].isin(models_plot)]
              .groupby('model')['mean'].mean())
    ann_lines = ['Pooled mean Δ NLL:']
    for m, lbl in zip(models_plot, labels_plot):
        v = pooled.get(m, float('nan'))
        ann_lines.append(f'  {lbl}: {v:+.4f}')
    ax.text(0.98, 0.97, chr(10).join(ann_lines),
            transform=ax.transAxes, va='top', ha='right',
            fontsize=FONTSIZE - 3, family='monospace',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor='#cccccc', alpha=0.9))

    fig.tight_layout()
    fig_v1 = FIGURES_VOL_DIR / 'fig_V1_headtohead'
    fig.savefig(str(fig_v1) + '.pdf'); fig.savefig(str(fig_v1) + '.png')
    plt.close(fig)
    print(f'  Saved {fig_v1}.pdf/.png')


  Saved figures\vol\fig_V1_headtohead.pdf/.png


## Fig V2 — Vol window comparison, JPM (all h, all N)

In [15]:
if not vol_data_available or nll_vol_all.empty:
    print('SKIP: results_vol/ not available')
else:
    vol_models = ['state_cond_vol_w5', 'state_cond_vol_w10', 'state_cond_vol_w21']
    colors_w = {'state_cond_vol_w5': '#ff7f0e', 'state_cond_vol_w10': '#2ca02c', 'state_cond_vol_w21': '#d62728'}
    baseline_m = 'state_cond_return'
    jpm = nll_vol_all[nll_vol_all['ticker'].eq('JPM')]

    fig, axes = plt.subplots(1, len(BINS_VOL), figsize=(DOUBLE_COL * 1.5, 3.5), sharey=True)
    for ax, N_plot in zip(axes, BINS_VOL):
        sub = jpm[jpm['N'].eq(N_plot)]
        for m in [baseline_m] + vol_models:
            grp = sub[sub['model'].eq(m)].groupby('h')['nll_test'].mean()
            lbl = m.replace('state_cond_', '')
            c = '#1f77b4' if m == baseline_m else colors_w[m]
            ls = '--' if m == baseline_m else '-'
            ax.plot(grp.index, grp.values, marker='o', color=c, ls=ls, label=lbl)
        ax.set_title(f'N={N_plot}')
        ax.set_xlabel('Horizon h')
    axes[0].set_ylabel('Test NLL')
    axes[0].legend(fontsize=FONTSIZE - 2)
    fig.suptitle('JPM — vol window comparison')
    fig.tight_layout()
    fig_v2 = FIGURES_VOL_DIR / 'fig_V2_vol_window_JPM'
    fig.savefig(str(fig_v2) + '.pdf'); fig.savefig(str(fig_v2) + '.png')
    plt.close(fig)
    print(f'  Saved {fig_v2}.pdf/.png')


  Saved figures\vol\fig_V2_vol_window_JPM.pdf/.png


## Fig V3 — σ_t vs row entropy scatter, JPM (h=1, N=55, seed=42)

In [16]:
if not vol_data_available or nll_vol_all.empty:
    print('SKIP: results_vol/ not available')
else:
    import numpy as np
    from scipy import stats

    op_path = VOL_RESULTS_DIR / 'JPM' / 'state_cond_vol_w21' / 'operator_ts_h1_N55_seed42.csv'
    if not op_path.exists():
        print(f'SKIP Fig V3: {op_path} not found')
    elif 'JPM' not in vol_state_ts:
        print('SKIP Fig V3: vol_state_ts[JPM] not loaded')
    else:
        op_df  = pd.read_csv(op_path, parse_dates=['date'])
        ts_jpm = vol_state_ts['JPM']
        merged = op_df.merge(ts_jpm[['date', 'sigma_t_w21']], on='date', how='inner')
        merged = merged.dropna(subset=['sigma_t_w21', 'row_entropy'])

        x_vals = merged['sigma_t_w21'].values
        y_vals = merged['row_entropy'].values
        t_idx  = np.arange(len(merged))   # ordinal time index for color

        # Pearson correlation & linear regression
        r_val, p_val = stats.pearsonr(x_vals, y_vals)
        slope, intercept, *_ = stats.linregress(x_vals, y_vals)

        # 95% CI band via bootstrap (fast: analytical SE of prediction)
        x_line = np.linspace(x_vals.min(), x_vals.max(), 200)
        y_line = intercept + slope * x_line
        n      = len(x_vals)
        x_mean = x_vals.mean()
        s_err  = np.sqrt(np.sum((y_vals - (intercept + slope * x_vals))**2) / (n - 2))
        se_band = s_err * np.sqrt(1/n + (x_line - x_mean)**2 / np.sum((x_vals - x_mean)**2))
        y_lo = y_line - 1.96 * se_band
        y_hi = y_line + 1.96 * se_band

        fig, ax = plt.subplots(figsize=(SINGLE_COL * 1.8, SINGLE_COL * 1.6))
        sc = ax.scatter(x_vals, y_vals, c=t_idx, cmap='viridis',
                        s=4, alpha=0.55, linewidths=0, zorder=3)
        cbar = fig.colorbar(sc, ax=ax, pad=0.02)
        cbar.set_label('Time (test period, early→late)', fontsize=FONTSIZE - 2)
        cbar.ax.tick_params(labelsize=FONTSIZE - 3)

        ax.plot(x_line, y_line, color='#d62728', lw=1.5, zorder=4, label='OLS fit')
        ax.fill_between(x_line, y_lo, y_hi, color='#d62728', alpha=0.15, zorder=2, label='95% CI')

        p_str = f'p<0.001' if p_val < 0.001 else f'p={p_val:.3f}'
        ax.set_xlabel('Realized volatility σ_t (w=21)')
        ax.set_ylabel('Row entropy')
        ax.set_title(f'JPM: σ_t vs row entropy\nr={r_val:.3f}, {p_str} (h=1, N=55)')
        ax.legend(fontsize=FONTSIZE - 2)
        fig.tight_layout()
        fig_v3 = FIGURES_VOL_DIR / 'fig_V3_vol_entropy_scatter'
        fig.savefig(str(fig_v3) + '.pdf'); fig.savefig(str(fig_v3) + '.png')
        plt.close(fig)
        print(f'  Saved {fig_v3}.pdf/.png')


  Saved figures\vol\fig_V3_vol_entropy_scatter.pdf/.png


## Fig V4 — Operator snapshots: vol-state vs return-state side by side (h=1, N=55, seed=42)

In [17]:
if not vol_data_available or nll_vol_all.empty:
    print('SKIP: results_vol/ not available')
else:
    import numpy as np
    import matplotlib.colors as mcolors

    snap_return = VOL_RESULTS_DIR / 'JPM' / 'state_cond_return'  / 'snapshots'
    snap_vol    = VOL_RESULTS_DIR / 'JPM' / 'state_cond_vol_w21' / 'snapshots'
    snap_labels = ['calm', 'bearish', 'bullish']

    return_snaps = {l: np.load(snap_return / f'operator_snapshot_{l}_seed42.npy')
                    for l in snap_labels
                    if (snap_return / f'operator_snapshot_{l}_seed42.npy').exists()}
    vol_snaps    = {l: np.load(snap_vol    / f'operator_snapshot_{l}_seed42.npy')
                    for l in snap_labels
                    if (snap_vol    / f'operator_snapshot_{l}_seed42.npy').exists()}

    if not return_snaps or not vol_snaps:
        print('SKIP Fig V4: snapshot files not found')
    else:
        vmax = 0.05
        tick_bins = [0, 10, 20, 30, 40, 50]
        cmap_ret = 'Blues'
        cmap_vol = 'Purples'

        # Leave room for colorbars on right using constrained_layout
        fig, axes = plt.subplots(2, 3, figsize=(DOUBLE_COL * 1.05, 5.0),
                                 constrained_layout=True)

        ims_ret, ims_vol = [], []
        for col, lbl in enumerate(snap_labels):
            ax_r = axes[0, col]
            ax_v = axes[1, col]

            # Return-state row
            if lbl in return_snaps:
                im_r = ax_r.imshow(return_snaps[lbl], aspect='auto',
                                   vmin=0, vmax=vmax, cmap=cmap_ret, origin='upper')
                ims_ret.append(im_r)
            ax_r.set_title(f'Return-state\n{lbl}', fontsize=FONTSIZE - 1)

            # Vol-state row
            if lbl in vol_snaps:
                im_v = ax_v.imshow(vol_snaps[lbl], aspect='auto',
                                   vmin=0, vmax=vmax, cmap=cmap_vol, origin='upper')
                ims_vol.append(im_v)
            ax_v.set_title(f'Vol-state\n{lbl}', fontsize=FONTSIZE - 1)

            # X-axis ticks + labels on bottom row
            ax_v.set_xticks(tick_bins)
            ax_v.set_xticklabels([str(t) for t in tick_bins], fontsize=FONTSIZE - 3)
            ax_v.set_xlabel('Output bin y', fontsize=FONTSIZE - 2)

            # Clear x-ticks on top row
            ax_r.set_xticks([])

            # Y-axis ticks on all panels (left-most gets label)
            for ax in (ax_r, ax_v):
                ax.set_yticks(tick_bins)
                ax.set_yticklabels([str(t) for t in tick_bins], fontsize=FONTSIZE - 3)

        # Y-axis labels on leftmost panels only
        axes[0, 0].set_ylabel('Input state X_t (return bin)', fontsize=FONTSIZE - 2)
        axes[1, 0].set_ylabel('Input state σ_t (vol bin)',    fontsize=FONTSIZE - 2)

        # Clear y-tick labels on non-leftmost panels
        for row in range(2):
            for col in range(1, 3):
                axes[row, col].set_yticklabels([])

        # Single shared colorbar (uses last im from either row, same norm)
        if ims_ret:
            cbar = fig.colorbar(ims_ret[-1], ax=axes[:, 2], shrink=0.85, pad=0.04)
            cbar.set_label('Predicted probability', fontsize=FONTSIZE - 2)
            cbar.ax.tick_params(labelsize=FONTSIZE - 3)

        fig.suptitle('Operator snapshots: return-state vs vol-state\n(JPM, h=1, N=55, seed=42)',
                     fontsize=FONTSIZE)
        fig_v4 = FIGURES_VOL_DIR / 'fig_V4_snapshots_comparison'
        fig.savefig(str(fig_v4) + '.pdf'); fig.savefig(str(fig_v4) + '.png')
        plt.close(fig)
        print(f'  Saved {fig_v4}.pdf/.png')


  Saved figures\vol\fig_V4_snapshots_comparison.pdf/.png


## Fig V5 — Cross-asset synchrony: vol-state model, row entropy per ticker (h=1, N=55, seed=42)

In [18]:
if not vol_data_available or nll_vol_all.empty:
    print('SKIP: results_vol/ not available')
else:
    frames_v5 = []
    for t in TICKERS:
        p = VOL_RESULTS_DIR / t / 'state_cond_vol_w21' / 'operator_ts_h1_N55_seed42.csv'
        if p.exists():
            df = pd.read_csv(p, parse_dates=['date'])
            df['ticker'] = t
            frames_v5.append(df)

    if not frames_v5:
        print('SKIP Fig V5: no operator_ts CSVs found')
    else:
        dfv5 = pd.concat(frames_v5, ignore_index=True)
        fig, ax = plt.subplots(figsize=(DOUBLE_COL, 3.5))
        for t, grp in dfv5.groupby('ticker'):
            c = TICKER_COLORS.get(t, 'gray')
            ax.plot(grp['date'], grp['row_entropy'].rolling(21, min_periods=1).mean(),
                    color=c, lw=0.7, alpha=0.9, label=t)
        shade_stress(ax)
        ax.set_ylabel('Row entropy (21-day MA)')
        ax.set_xlabel('Date')
        ax.set_title('Cross-asset synchrony — vol-state model (h=1, N=55)')
        ax.legend(fontsize=FONTSIZE - 3, ncol=2)
        fig.tight_layout()
        fig_v5 = FIGURES_VOL_DIR / 'fig_V5_crossasset_vol'
        fig.savefig(str(fig_v5) + '.pdf'); fig.savefig(str(fig_v5) + '.png')
        plt.close(fig)
        print(f'  Saved {fig_v5}.pdf/.png')


  Saved figures\vol\fig_V5_crossasset_vol.pdf/.png


## Fig V6 — NLL vs horizon: vol-state vs return-state, 10-stock grid (N=55)

In [19]:
if not vol_data_available or nll_vol_all.empty:
    print('SKIP: results_vol/ not available')
else:
    import math

    models_v6 = {
        'state_cond_return':   ('#1f77b4', '--'),
        'state_cond_vol_w21':  ('#9467bd', '-'),
        'state_free':          ('#7f7f7f', ':'),
    }
    sub_v6 = nll_vol_all[nll_vol_all['N'].eq(55)]

    ncols = 5
    nrows = math.ceil(len(TICKERS) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(DOUBLE_COL * 1.4, nrows * 2.4),
                             sharey=False, sharex=True)
    axes_flat = axes.flat
    for ax, t in zip(axes_flat, TICKERS):
        grp = sub_v6[sub_v6['ticker'].eq(t)]
        for m, (c, ls) in models_v6.items():
            row = grp[grp['model'].eq(m)].groupby('h')['nll_test'].mean()
            if not row.empty:
                ax.plot(row.index, row.values, color=c, ls=ls, marker='o',
                        ms=3, lw=1.2, label=m.replace('state_cond_', ''))
        ax.set_title(t, fontsize=FONTSIZE - 1)
        ax.set_xlabel('h')
    # Hide unused subplots
    for ax in list(axes_flat)[len(TICKERS):]:
        ax.set_visible(False)
    axes[0, 0].set_ylabel('Test NLL')
    handles = [plt.Line2D([0],[0], color=c, ls=ls, label=m.replace('state_cond_',''))
               for m, (c, ls) in models_v6.items()]
    fig.legend(handles=handles, loc='lower right', fontsize=FONTSIZE - 2, ncol=3)
    fig.suptitle('NLL vs horizon: vol-state vs return-state (N=55)')
    fig.tight_layout()
    fig_v6 = FIGURES_VOL_DIR / 'fig_V6_nll_horizon_grid'
    fig.savefig(str(fig_v6) + '.pdf'); fig.savefig(str(fig_v6) + '.png')
    plt.close(fig)
    print(f'  Saved {fig_v6}.pdf/.png')


  Saved figures\vol\fig_V6_nll_horizon_grid.pdf/.png


## Fig V7 — Per-stock paired comparison: return-state vs vol_w21 (slope chart, h=1, N=55)

In [20]:
if not vol_data_available or nll_vol_all.empty:
    print('SKIP: results_vol/ not available')
else:
    import numpy as np

    sub7 = nll_vol_all[
        nll_vol_all['h'].eq(1) & nll_vol_all['N'].eq(55)
        & nll_vol_all['model'].isin(['state_cond_return', 'state_cond_vol_w21'])
    ].copy()

    # Mean delta_marginal per (ticker, model) across seeds
    means7 = (sub7.groupby(['ticker', 'model'])['delta_marginal']
              .mean().reset_index()
              .pivot(index='ticker', columns='model', values='delta_marginal'))

    # Sort tickers by vol_w21 delta_marginal
    means7 = means7.sort_values('state_cond_vol_w21')

    ret_col = 'state_cond_return'
    vol_col = 'state_cond_vol_w21'

    # Classify: green = vol wins, red = return wins, gray = within 0.005
    THRESHOLD = 0.005
    def line_color(r, v):
        if (v - r) < -THRESHOLD:   return '#2ca02c'   # vol wins (more negative)
        if (r - v) < -THRESHOLD:   return '#d62728'   # return wins
        return '#aaaaaa'

    n_vol_wins = sum(
        (means7[vol_col].values[i] - means7[ret_col].values[i]) < -THRESHOLD
        for i in range(len(means7))
    )
    frac_vol = n_vol_wins / len(means7)

    fig, ax = plt.subplots(figsize=(SINGLE_COL * 1.6, SINGLE_COL * 2.0))

    x0, x1 = 0.0, 1.0
    for ticker, row_7 in means7.iterrows():
        r_val = row_7[ret_col]
        v_val = row_7[vol_col]
        if np.isnan(r_val) or np.isnan(v_val):
            continue
        c = line_color(r_val, v_val)
        lw = 1.8 if c != '#aaaaaa' else 1.0
        ax.plot([x0, x1], [r_val, v_val], color=c, lw=lw, alpha=0.85, zorder=3)
        ax.text(x0 - 0.03, r_val, ticker, ha='right', va='center',
                fontsize=FONTSIZE - 3, color=c)
        ax.text(x1 + 0.03, v_val, ticker, ha='left', va='center',
                fontsize=FONTSIZE - 3, color=c)

    ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.5, zorder=2)
    ax.set_xlim(-0.35, 1.35)
    ax.set_xticks([x0, x1])
    ax.set_xticklabels(['return-state', 'vol_w21'], fontsize=FONTSIZE - 1)
    ax.set_ylabel('Δ NLL vs marginal (nats)')
    ax.set_title(
        f'Vol-state wins in {n_vol_wins}/{len(means7)} stocks '
        f'({frac_vol:.0%})\n'
        f'(h=1, N=55, mean over 3 seeds; green=vol wins, red=return wins)',
        fontsize=FONTSIZE - 1,
    )
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    fig.tight_layout()
    fig_v7 = FIGURES_VOL_DIR / 'fig_V7_slope_chart'
    fig.savefig(str(fig_v7) + '.pdf'); fig.savefig(str(fig_v7) + '.png')
    plt.close(fig)
    print(f'  Saved {fig_v7}.pdf/.png')


  Saved figures\vol\fig_V7_slope_chart.pdf/.png


## Manifest + LaTeX Tables

In [21]:
import math

# ── Figure manifest ──────────────────────────────────────────────────────────
print('=' * 55)
print('FIGURE MANIFEST')
print('=' * 55)
for fig_stem in [f'fig{i}' for i in range(1,13)] + ['fig5a','fig5b']:
    p = FIGURES_DIR / (fig_stem + '.pdf')
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  {fig_stem:8s}: {status}')

# ── LaTeX Table 1 — Baseline NLL ─────────────────────────────────────────────
def _tex_header(caption, label):
    return (
        r'\documentclass{article}' + '\n'
        r'\usepackage{booktabs}' + '\n'
        r'\begin{document}' + '\n'
        r'\begin{table}[ht]' + '\n'
        r'\centering' + '\n'
        f'\\caption{{{caption}}}\n'
        f'\\label{{{label}}}\n'
    )
def _tex_footer():
    return (
        r'\end{table}' + '\n'
        r'\end{document}'
    )

def _df_to_booktabs(df):
    cols = list(df.columns)
    col_fmt = 'l' + 'r' * (len(cols) - 1)
    lines  = [f'\\begin{{tabular}}{{{col_fmt}}}', r'\toprule']
    lines += [' & '.join(str(c) for c in cols) + r' \\']
    lines += [r'\midrule']
    for _, row in df.iterrows():
        lines.append(' & '.join(
            (f'{v:.4f}' if isinstance(v, float) else str(v)) for v in row
        ) + r' \\')
    lines += [r'\bottomrule', r'\end{tabular}']
    return '\n'.join(lines)

# Table 1 — Baseline NLL (A-EXT, state_cond, N=55)
if not nll_A_ext.empty:
    t1 = nll_A_ext[(nll_A_ext.N==55) & (nll_A_ext.model=='state_cond')]\
         .groupby(['ticker','h'])[['nll_test','delta_marginal']].mean()\
         .reset_index().round(4)
    tex1 = (_tex_header('Baseline NLL (state-cond, N=55)', 'tab:baseline-nll')
            + _df_to_booktabs(t1) + '\n' + _tex_footer())
    (TABLES_DIR / 'Table_1_baseline_nll.tex').write_text(tex1)
    print('\nTable_1_baseline_nll.tex written')

# Table 2 — Best config per stock
if not nll_C_ext.empty:
    best_per = []
    for t in TICKERS:
        sub = nll_C_ext[nll_C_ext.ticker==t].dropna(subset=['delta_marginal'])
        if sub.empty: continue
        bi = sub.groupby(['model','h','N'])['delta_marginal'].mean().idxmin()
        row = {'ticker': t, 'model': bi[0], 'h': bi[1], 'N': bi[2],
               'delta_marginal': sub.groupby(['model','h','N'])['delta_marginal'].mean()[bi]}
        best_per.append(row)
    if best_per:
        t2 = pd.DataFrame(best_per).round(4)
        tex2 = (_tex_header('Best configuration per stock (C-EXT)', 'tab:best-config')
                + _df_to_booktabs(t2) + '\n' + _tex_footer())
        (TABLES_DIR / 'Table_2_best_config_per_stock.tex').write_text(tex2)
        print('Table_2_best_config_per_stock.tex written')

# Table 3 — Regime comparison (h=1, N=55)
if not nll_E_ext.empty:
    t3 = nll_E_ext[(nll_E_ext.h==1) & (nll_E_ext.N==55)]\
         .groupby(['ticker','model'])['delta_marginal'].mean()\
         .reset_index().round(4)
    t3['regime'] = t3['model'].str.replace('state_cond_', '', regex=False)
    t3 = t3.pivot(index='ticker', columns='regime', values='delta_marginal').reset_index()
    tex3 = (_tex_header('Conditioning regime comparison (h=1, N=55)', 'tab:regime')
            + _df_to_booktabs(t3) + '\n' + _tex_footer())
    (TABLES_DIR / 'Table_3_regime_comparison.tex').write_text(tex3)
    print('Table_3_regime_comparison.tex written')

# Table 4 — Higher-order k
if not nll_D_ext.empty:
    nll_D_ext['k'] = nll_D_ext['model'].str.extract(r'ho_k(\d+)').astype(int)
    t4 = nll_D_ext.groupby(['k','h'])[['nll_test','delta_marginal']].mean()\
         .reset_index().round(4)
    tex4 = (_tex_header('Higher-order Markov ablation (JPM, N=55)', 'tab:higher-order')
            + _df_to_booktabs(t4) + '\n' + _tex_footer())
    (TABLES_DIR / 'Table_4_higher_order_k.tex').write_text(tex4)
    print('Table_4_higher_order_k.tex written')

# Table 5 — Sigma sweep
if not sigma_h1.empty:
    t5 = sigma_h1.groupby('sigma')[['nll_val','nll_test']].mean().reset_index().round(4)
    tex5 = (_tex_header('Sigma sweep on JPM (h=1, N=55)', 'tab:sigma')
            + _df_to_booktabs(t5) + '\n' + _tex_footer())
    (TABLES_DIR / 'Table_5_sigma_sweep.tex').write_text(tex5)
    print('Table_5_sigma_sweep.tex written')

print('\nDone.')
# ── Vol figure manifest ───────────────────────────────────────────────────────
if vol_data_available:
    vol_fig_stems = [
        'fig_V1_headtohead', 'fig_V2_vol_window_JPM', 'fig_V3_vol_entropy_JPM',
        'fig_V4_snapshots_comparison', 'fig_V5_crossasset_vol', 'fig_V6_nll_horizon_grid',
        'fig_V7_slope_chart',
    ]
    for stem in vol_fig_stems:
        p = FIGURES_VOL_DIR / (stem + '.pdf')
        status = 'OK' if p.exists() else 'MISSING'
        print(f'  {status}: figures/vol/{stem}.pdf')
else:
    print('  SKIPPED: vol figures (results_vol/ not found)')


FIGURE MANIFEST
  fig1    : OK
  fig2    : OK
  fig3    : OK
  fig4    : OK
  fig5    : MISSING
  fig6    : OK
  fig7    : OK
  fig8    : OK
  fig9    : OK
  fig10   : OK
  fig11   : OK
  fig12   : OK
  fig5a   : OK
  fig5b   : OK

Table_1_baseline_nll.tex written
Table_2_best_config_per_stock.tex written
Table_3_regime_comparison.tex written
Table_4_higher_order_k.tex written
Table_5_sigma_sweep.tex written

Done.
  OK: figures/vol/fig_V1_headtohead.pdf
  OK: figures/vol/fig_V2_vol_window_JPM.pdf
  OK: figures/vol/fig_V3_vol_entropy_JPM.pdf
  OK: figures/vol/fig_V4_snapshots_comparison.pdf
  OK: figures/vol/fig_V5_crossasset_vol.pdf
  OK: figures/vol/fig_V6_nll_horizon_grid.pdf
  OK: figures/vol/fig_V7_slope_chart.pdf
